# Lags, leads and moving averages

Adds time-shifted copies of the daily panel's columns. Reads `data/processed/daily_panel.parquet` and its codebook (both written by `01_data_aggregation.ipynb`). Writes:

- `data/processed/daily_panel_lagged.parquet`: the original columns plus the derived columns below.
- `data/processed/daily_panel_lagged_codebook.csv`: the original codebook plus one row per derived column, with `base_column`, `transform` and `days`.

Reports go to `outputs/02_lag_variables/`. The analysis notebook (`03_analysis.ipynb`) loads these two files.

| Derived column | Columns | Value on day *t* | Purpose |
|---|---|---|---|
| `x_lag<k>`, *k* = 1, 2, 3, 7, 14 | every column | `x` on day *t − k* | single-lag sensitivity models, cross-correlation checks, forecasting |
| `x_lead<k>`, *k* = 1, 2, 3, 7, 14 | primary exposures | `x` on day *t + k* | **negative control** (future exposure, preregistered) |
| `x_ma0_<k>`, *k* = 1, 2, 3, 7 | daily temperature and air pollution columns (°C, µg/m³) | mean of `x` on days *t − k* … *t* | moving-average exposure, sensitivity analyses |

**What these columns are not.** The main model is a quasi-Poisson DLNM with temperature lags to about 21 days (preregistration). A distributed-lag model needs every lag from 0 to *L*; it builds its lag basis inside the model (`crossbasis` in R `dlnm`, or the Python equivalent), not from these columns. The preregistered cross-correlation is computed on prewhitened residuals, so it needs lags of the residuals, formed in the analysis step.

**How they are formed.** The panel has one row per calendar day with no gaps (checked below), so *k* rows = *k* days. Days outside the panel are unknown: the first *k* days of a lag and the last *k* days of a lead are `NaN`. A moving average is `NaN` unless every day in its window is present (no partial means). Nothing is filled in.

**Weekly exposures** (`ili_rate`, `flu_index`) hold one value per ISO week, so leads under 7 days mostly repeat the current week; only leads of 7 and 14 days reach a different week.

**Structural breaks.** Lags and moving averages cross the breaks flagged in the panel (`triage_break_2024_06`, `hp_referral_era`): for up to 14 days after a break, a lagged outcome mixes old and new definitions.

**All data stays local.** `data/` and `outputs/` are gitignored. Do not print day-level rows.

## Imports

In [ ]:
import hashlib
import json
import platform
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version as pkg_version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Configuration

In [ ]:
# Project root: the folder with requirements-lock.txt (works when run from notebooks/ or from the root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements-lock.txt").exists())
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs" / "02_lag_variables"
OUTPUTS.mkdir(parents=True, exist_ok=True)

PANEL_IN = PROCESSED / "daily_panel.parquet"
CODEBOOK_IN = PROCESSED / "daily_panel_codebook.csv"
RUN_INFO_01 = ROOT / "outputs" / "01_data_aggregation" / "run_info.json"   # records the checksum of the panel 01 wrote
PANEL_OUT = PROCESSED / "daily_panel_lagged.parquet"
CODEBOOK_OUT = PROCESSED / "daily_panel_lagged_codebook.csv"

LAGS = [1, 2, 3, 7, 14]        # days, every column
LEADS = [1, 2, 3, 7, 14]       # days, primary exposures (negative control)
MA_WINDOWS = [1, 2, 3, 7]      # moving average over days t-k .. t
MA_SOURCES = {"arso_weather", "era5", "arso_air"}
MA_UNITS = {"°C", "µg/m³"}     # continuous temperature and pollutant columns

plt.rcParams.update({
    "figure.figsize": (12, 3.6), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False, "font.size": 9,
})

## 1. Load and check the panel

The run stops if:

- the panel is not the file `01_data_aggregation.ipynb` last wrote (its SHA-256 differs from `panel_sha256` in that notebook's `run_info.json`);
- the dates are not one row per consecutive calendar day;
- the panel and codebook do not list the same columns, or a derived column name already exists.

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


run_info_01 = json.loads(RUN_INFO_01.read_text())
panel_sha = sha256(PANEL_IN)
if panel_sha != run_info_01["panel_sha256"]:
    raise RuntimeError(f"{PANEL_IN.name} differs from the panel recorded in {RUN_INFO_01.relative_to(ROOT)} "
                       "(run 01_data_aggregation.ipynb again).")

panel = pd.read_parquet(PANEL_IN)
codebook = pd.read_csv(CODEBOOK_IN)
START, END = panel.index.min(), panel.index.max()

if not panel.index.equals(pd.date_range(START, END, freq="D", name="date")):
    raise RuntimeError("Panel index is not one row per consecutive calendar day: shifts in rows would not be shifts in days.")
if set(codebook["column"]) != set(panel.columns):
    raise RuntimeError(f"Panel and codebook differ: only in panel {sorted(set(panel.columns) - set(codebook['column']))}, "
                       f"only in codebook {sorted(set(codebook['column']) - set(panel.columns))}")

cb = codebook.set_index("column")
LEAD_COLUMNS = cb.index[(cb["use"] == "primary") & ~cb["source"].isin(["outcomes", "calendar"])].tolist()
MA_COLUMNS = cb.index[cb["source"].isin(MA_SOURCES) & cb["unit"].isin(MA_UNITS) & (cb["use"] != "technical")].tolist()

print(f"Panel: {len(panel)} days ({START.date()} to {END.date()}), {panel.shape[1]} columns; checksum matches 01's run record")
print(f"Lags {LAGS}: all {panel.shape[1]} columns")
print(f"Leads {LEADS}: {LEAD_COLUMNS}")
print(f"Moving averages 0-{MA_WINDOWS}: {len(MA_COLUMNS)} columns: {MA_COLUMNS}")

## 2. Build the derived columns

Each base column is followed by its lags, leads and moving averages (`x`, `x_lag1`, …, `x_lead1`, …, `x_ma0_1`, …). Integer columns become float in derived columns, because of the `NaN` at the edges.

Every derived column is checked against an independent calculation on the raw arrays.

In [ ]:
SPEC = {}   # derived column -> (base column, transform, days)
for c in panel.columns:
    SPEC.update({f"{c}_lag{k}": (c, "lag", k) for k in LAGS})
    if c in LEAD_COLUMNS:
        SPEC.update({f"{c}_lead{k}": (c, "lead", k) for k in LEADS})
    if c in MA_COLUMNS:
        SPEC.update({f"{c}_ma0_{k}": (c, "mean", k) for k in MA_WINDOWS})

clash = sorted(set(SPEC) & set(panel.columns))
if clash:
    raise RuntimeError(f"Derived column names already exist in the panel: {clash}")


def derive(base: str, transform: str, k: int) -> pd.Series:
    x = panel[base]
    if transform == "lag":
        return x.shift(k)
    if transform == "lead":
        return x.shift(-k)
    return x.astype(float).rolling(k + 1, min_periods=k + 1).mean()


derived = pd.DataFrame({name: derive(*spec) for name, spec in SPEC.items()}, index=panel.index)
order = [c for b in panel.columns for c in [b, *(n for n, s in SPEC.items() if s[0] == b)]]
out = pd.concat([panel, derived], axis=1)[order]

# Independent check on the raw arrays
for name, (base, transform, k) in SPEC.items():
    x = panel[base].to_numpy(dtype=float)
    expected = np.full_like(x, np.nan)
    if transform == "lag":
        expected[k:] = x[:-k]
    elif transform == "lead":
        expected[:-k] = x[k:]
    else:
        win = np.lib.stride_tricks.sliding_window_view(x, k + 1)
        expected[k:] = win.mean(axis=1)   # NaN anywhere in the window -> NaN
    if not np.allclose(out[name].to_numpy(dtype=float), expected, equal_nan=True, rtol=0, atol=1e-9):
        raise RuntimeError(f"Check failed for {name}")

counts = pd.Series([s[1] for s in SPEC.values()]).value_counts()
print(f"Derived panel: {out.shape[0]} days x {out.shape[1]} columns "
      f"({counts.get('lag', 0)} lags, {counts.get('lead', 0)} leads, {counts.get('mean', 0)} moving averages); all checked")

## 3. Codebook for the derived columns

Each derived column gets a codebook row copied from its base column (source, unit, version, note), with:

- `base_column`, `transform` (`lag`, `lead`, `mean`) and `days` (*k*); empty / `none` / 0 for the original columns;
- `definition`: what the transform does, followed by the base definition;
- `forecast_lag_days`: lag: base − *k*; lead: base + *k*; mean: base (the window ends on day *t*). Empty stays empty (not a real-time series);
- `valid_from`, `valid_to`: shifted by the transform and kept within the panel;
- `pct_missing`, `n_distinct`, `n_nonzero_days`: recomputed on the derived column.

**`use` of derived columns.** A derived column does not inherit `primary`: the primary models build their own lag structure, so copying the label would put e.g. `arr_all_mts_all_lag7` among the primary variables. First matching rule wins:

| Base column | Transform | `use` | `hypothesis` |
|---|---|---|---|
| primary exposure | lead | `negative_control` (new value) | kept |
| `technical`, or source `calendar` (`time`, `dow`, `month`, holidays, …) | any | `technical` | cleared |
| outcome (source `outcomes`), `descriptive` | any | `descriptive` | cleared |
| outcome (source `outcomes`) | lag | `forecasting` (autoregressive terms; not for explanatory models) | cleared |
| `primary` or `secondary` | lag, mean | `sensitivity` | kept |
| anything else | lag, mean | same as base | kept |

In [ ]:
def clip_date(d: pd.Timestamp) -> str:
    return min(max(d, START), END).date().isoformat()


def derived_use(base: pd.Series, transform: str) -> tuple[str, object]:
    if transform == "lead":
        return "negative_control", base["hypothesis"]
    if base["use"] == "technical" or base["source"] == "calendar":
        return "technical", pd.NA
    if base["source"] == "outcomes":
        return ("descriptive" if base["use"] == "descriptive" else "forecasting"), pd.NA
    if base["use"] in ("primary", "secondary"):
        return "sensitivity", base["hypothesis"]
    return base["use"], base["hypothesis"]


def describe(s: pd.Series) -> dict:
    return {"pct_missing": round(s.isna().mean() * 100, 1), "n_distinct": s.nunique(),
            "n_nonzero_days": int((s.fillna(0) != 0).sum())}


TEXT = {
    "lag": lambda c, k: f"Value of `{c}` {k} day{'s' * (k > 1)} earlier (day t − {k}).",
    "lead": lambda c, k: f"Value of `{c}` {k} day{'s' * (k > 1)} later (day t + {k}). Negative control: a future exposure "
                         "cannot cause today's arrivals, so an association indicates residual confounding.",
    "mean": lambda c, k: f"Mean of `{c}` over days t − {k} to t ({k + 1} days); missing if any day in the window is missing.",
}
SHIFT = {"lag": lambda k: (k, k), "lead": lambda k: (-k, -k), "mean": lambda k: (k, 0)}   # days added to (valid_from, valid_to)
FLD = {"lag": lambda v, k: v - k, "lead": lambda v, k: v + k, "mean": lambda v, k: v}

rows = []
base_cb = cb.assign(base_column=pd.NA, transform="none", days=0)
for col in out.columns:
    if col not in SPEC:
        rows.append(base_cb.loc[col].rename(col))
        continue
    c, transform, k = SPEC[col]
    b = cb.loc[c]
    r = base_cb.loc[c].copy()
    r["base_column"], r["transform"], r["days"] = c, transform, k
    r["definition"] = f"{TEXT[transform](c, k)} {b['definition']}"
    r["use"], r["hypothesis"] = derived_use(b, transform)
    if pd.notna(b["forecast_lag_days"]):
        r["forecast_lag_days"] = FLD[transform](b["forecast_lag_days"], k)
    if pd.notna(b["available"]):
        r["available"] = f"{transform} {k} d of base column: {b['available']}"
    d_from, d_to = SHIFT[transform](k)
    r["valid_from"] = clip_date(pd.Timestamp(b["valid_from"]) + pd.Timedelta(days=d_from)) if pd.notna(b["valid_from"]) else pd.NA
    r["valid_to"] = clip_date(pd.Timestamp(b["valid_to"]) + pd.Timedelta(days=d_to)) if pd.notna(b["valid_to"]) else pd.NA
    r.update(pd.Series(describe(out[col])))
    rows.append(r.rename(col))

lag_codebook = pd.DataFrame(rows).rename_axis("column").reset_index()
lag_codebook["days"] = lag_codebook["days"].astype(int)

# Checks
if list(lag_codebook["column"]) != list(out.columns):
    raise RuntimeError("Codebook rows do not match the derived panel's columns")
is_derived = lag_codebook["transform"] != "none"
if (lag_codebook.loc[is_derived, "use"] == "primary").any():
    raise RuntimeError("A derived column is labelled primary")
vt = pd.to_datetime(lag_codebook["valid_to"])
vf = pd.to_datetime(lag_codebook["valid_from"])
if (vt > END).any() or (vf < START).any():
    raise RuntimeError("valid_from / valid_to outside the panel")

display(pd.crosstab(lag_codebook["use"], lag_codebook["transform"], margins=True))
with pd.option_context("display.max_colwidth", 80):
    display(lag_codebook.loc[lag_codebook["column"].str.startswith("temp_mean"),
                             ["column", "transform", "days", "use", "hypothesis", "forecast_lag_days", "valid_from", "valid_to"]])

## 4. Forecasting availability

How many columns are known at the end of day *t* (`forecast_lag_days` ≤ 0), by transform and days. Columns with an empty `forecast_lag_days` are not real-time series and are not counted. Leads are never known in advance, except for calendar-type columns, which have no leads here.

In [ ]:
fl = lag_codebook.dropna(subset=["forecast_lag_days"])
avail = (fl.assign(known=fl["forecast_lag_days"] <= 0)
           .groupby(["transform", "days"])["known"].agg(known="sum", total="size"))
avail["pct_known"] = (avail["known"] / avail["total"] * 100).round(1)
avail.to_csv(OUTPUTS / "forecast_availability.csv")
display(avail)

## 5. Autocorrelation of the primary outcomes (descriptive only)

Pearson correlation of each primary outcome with its own lags, on the raw series. Both outcomes have a strong weekly cycle (weekends are much lower), which produces peaks at lags 7 and 14 and can make short lags negative. This is a description of the data, not an input to any model; the preregistered cross-correlations use prewhitened series.

In [ ]:
primary_outcomes = codebook.loc[(codebook["use"] == "primary") & (codebook["source"] == "outcomes"), "column"].tolist()
acf = pd.DataFrame({c: {k: out[c].corr(out[f"{c}_lag{k}"]) for k in LAGS} for c in primary_outcomes}).rename_axis("lag_days")
acf.round(3).to_csv(OUTPUTS / "primary_outcome_autocorrelation.csv")
display(acf.round(3))

fig, ax = plt.subplots(figsize=(8, 3.2))
for c in acf.columns:
    ax.plot(acf.index, acf[c], marker="o", label=c)
ax.axhline(0, color="grey", lw=0.8)
ax.set_xticks(LAGS)
ax.set_xlabel("Lag (days)")
ax.set_ylabel("Correlation with own lag")
ax.set_title("Autocorrelation of primary outcomes (raw series)")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUTS / "primary_outcome_autocorrelation.png")
plt.show()

## 6. Save and run record

`outputs/02_lag_variables/run_info.json` records the run in the same format as `01_data_aggregation.ipynb`: date, git commit, Python and package versions, the settings, and the checksums of the input and output files.

In [ ]:
def git(*args) -> str:
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:  # noqa: BLE001
        return "unavailable"


out.to_parquet(PANEL_OUT)
lag_codebook.to_csv(CODEBOOK_OUT, index=False)

PACKAGES = ["pandas", "numpy", "pyarrow", "matplotlib"]
run_info = {
    "run_at": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "git_commit": git("rev-parse", "HEAD"),
    "git_uncommitted_changes": bool(git("status", "--porcelain")),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {p: pkg_version(p) for p in PACKAGES},
    "settings": {"lags": LAGS, "leads": LEADS, "ma_windows": MA_WINDOWS,
                 "lead_columns": LEAD_COLUMNS, "ma_columns": MA_COLUMNS},
    "input_panel_sha256": panel_sha,
    "input_codebook_sha256": sha256(CODEBOOK_IN),
    "input_run_01": {"run_at": run_info_01["run_at"], "git_commit": run_info_01["git_commit"]},
    "panel_shape": list(out.shape),
    "panel_sha256": sha256(PANEL_OUT),
    "codebook_sha256": sha256(CODEBOOK_OUT),
}
(OUTPUTS / "run_info.json").write_text(json.dumps(run_info, indent=2, ensure_ascii=False))
print(f"Saved {PANEL_OUT.relative_to(ROOT)} {out.shape}, {CODEBOOK_OUT.relative_to(ROOT)}, "
      f"and {(OUTPUTS / 'run_info.json').relative_to(ROOT)}")